In [1]:
# Force install by calling the files directly and ignoring compatibility tags
!pip install /kaggle/input/datasets/pjleek/onnxruntime126-312/onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl --no-deps --force-reinstall


Processing /kaggle/input/datasets/pjleek/onnxruntime126-312/onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 1.3 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import glob
import math
import numpy as np
import os
from collections import namedtuple

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
CENTER_X, CENTER_Y = 50.0, 50.0
MAX_SPEED = 6.0

# ============================================================
# 1. The Value-Estimator Neural Network (OrbitNet V2)
# ============================================================
class OrbitNet(nn.Module):
    def __init__(self):
        super(OrbitNet, self).__init__()
        self.fc1 = nn.Linear(12, 128) # Expanded to 12 features
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        self.relu = nn.LeakyReLU(0.1) 
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.sigmoid(self.fc4(x))

# ============================================================
# 2. Physics & Future State Parsing
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def get_target_pos(tgt, turns, ang_vel, initial_planets, comets, comet_ids):
    if tgt.id in comet_ids:
        for c in comets:
            if tgt.id in c.get("planet_ids",[]):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + turns
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]): return c["paths"][idx][f_idx]
        return None
    init = initial_planets.get(tgt.id)
    if not init: return (tgt.x, tgt.y)
    r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
    if r + init.radius >= 50.0: return (tgt.x, tgt.y)
    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + ang_vel * turns
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships, ang_vel, initial_planets, comets, comet_ids):
    speed = fleet_speed(ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    for _ in range(5): 
        flight_dist = max(0.0, math.hypot(tx - src.x, ty - src.y) - src.radius - 0.1 - tgt.radius)
        eta = flight_dist / speed
        pos = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, initial_planets, comets, comet_ids)
        if not pos: return None, 999
        tx, ty = pos
    return int(math.ceil(eta)), tx, ty, speed

# ============================================================
# 3. The Coach's Rubric (Composite Reward Shaping)
# ============================================================
def calculate_coach_score(is_winner, tgt_prod, speed, tactical_success):
    win_score = 0.4 if is_winner else 0.0
    prod_score = 0.3 * (tgt_prod / 5.0)
    speed_score = 0.2 * min(1.0, speed / 6.0)
    tactical_score = 0.1 if tactical_success else 0.0
    
    final_score = win_score + prod_score + speed_score + tactical_score
    
    if not tactical_success:
        if is_winner: final_score *= 0.5 
        else: final_score *= 0.1 
            
    return final_score

# ============================================================
# 4. Chunked Data Extraction & Hindsight Logic
# ============================================================
def extract_training_chunk(files_chunk):
    X_data, Y_data = [],[]
    
    for filepath in files_chunk:
        try:
            with open(filepath, 'r') as f: match = json.load(f)
            steps = match.get("steps", [])
            if not steps: continue
            
            # Winner ID extraction
            last_agents = steps[-1]
            rewards = [agent.get("reward", -1) for agent in last_agents]
            winner_id = rewards.index(max(rewards))
            
            # Safe boundary - lookahead requires buffering
            for step_idx, step in enumerate(steps[:-25]): 
                obs = step[0].get("observation", {})
                planets = [Planet(*p) for p in obs.get("planets", [])]
                ang_vel = obs.get("angular_velocity", 0.0)
                initial_planets = {Planet(*p).id: Planet(*p) for p in obs.get("initial_planets",[])}
                comets = obs.get("comets",[])
                comet_ids = set(obs.get("comet_planet_ids",[]))
                
                # Opportunity Map (Calculated once per turn)
                best_neutral = max([p.production for p in planets if p.owner == -1] +[0])
                
                for player_id in range(len(step)):
                    actions = step[player_id].get("action",[])
                    is_winner = (player_id == winner_id)
                    
                    # Enemy pressure calculation
                    enemy_near = sum(p.ships for p in planets if p.owner != player_id and p.owner != -1)
                    
                    for act in actions:
                        src_id, angle, ships_sent = act[0], act[1], max(1, act[2])
                        src_p = next((p for p in planets if p.id == src_id), None)
                        if not src_p: continue

                        # Reverse-engineer target
                        best_diff, tgt_p = float('inf'), None
                        for p in planets:
                            if p.id == src_id: continue
                            tgt_ang = math.atan2(p.y - src_p.y, p.x - src_p.x)
                            ang_diff = abs((angle - tgt_ang + math.pi) % (2 * math.pi) - math.pi)
                            if ang_diff < best_diff:
                                best_diff, tgt_p = ang_diff, p
                                
                        if best_diff > 0.2 or not tgt_p: continue 
                            
                        # Physics check
                        res = plan_flight(src_p, tgt_p, ships_sent, ang_vel, initial_planets, comets, comet_ids)
                        if not res: continue
                        eta, tx, ty, speed = res
                        if eta == 999: continue
                        
                        # Future tactical check (arrival turn)
                        future_turn = min(len(steps) - 1, step_idx + eta + 2)
                        future_obs = steps[future_turn][0].get("observation", {}).get("planets",[])
                        future_tgt = next((p for p in future_obs if p[0] == tgt_p.id), None)
                        tactical_success = (future_tgt is not None and future_tgt[1] == player_id)

                        # --- HINDSIGHT ANALYSIS ---
                        future_step = steps[step_idx + 15]
                        f_planets = [Planet(*p) for p in future_step[0]["observation"]["planets"]]
                        f_src = next((p for p in f_planets if p.id == src_id), None)
                        
                        # Blunder: Loser moved ships out and then lost their planet within 15 turns
                        blunder = (not is_winner and f_src and f_src.owner != player_id)
                        
                        # --- PHYSICAL & CONTEXTUAL CORE FEATURES ---
                        npv = (tgt_p.production * max(0, 500 - step_idx - eta)) / 1000.0
                        in_quad = 1.0 if (1 if src_p.x>50 else -1) == (1 if tgt_p.x>50 else -1) and (1 if src_p.y>50 else -1) == (1 if tgt_p.y>50 else -1) else 0.0
                        
                        feat =[
                            src_p.production / 5.0,                    # 1. Source value
                            ships_sent / max(1.0, float(src_p.ships)), # 2. Aggression ratio
                            step_idx / 500.0,                          # 3. Game time
                            enemy_near / 1000.0,                       # 4. Defensive pressure
                            best_neutral / 5.0,                        # 5. Opportunity cost
                            1.0 if is_winner else 0.0,                 # 6. Success context
                            npv,                                       # 7. Net Present Value
                            eta / 50.0,                                # 8. Time discount
                            tgt_p.production / 5.0,                    # 9. Target raw yield
                            1.0 if tgt_p.owner != -1 else 0.5,         # 10. Target ownership
                            in_quad,                                   # 11. Quadrant matching
                            speed / 6.0                                # 12. Normalized speed
                        ]
                        
                        # Reward Shaping
                        score = calculate_coach_score(is_winner, tgt_p.production, speed, tactical_success)
                        if blunder: score *= 0.1 # The "Should have stayed home" penalty
                        
                        X_data.append(feat)
                        Y_data.append([score])
                        
        except Exception: 
            continue
            
    if not X_data: return torch.empty(0, 12), torch.empty(0, 1)
    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(Y_data, dtype=torch.float32)

# ============================================================
# 5. Iterative Memory-Safe Model Training
# ============================================================
def train_and_export():
    dataset_path = "/kaggle/input/datasets/bovard/orbit-wars-top10-episodes-2026-05-04/episodes/episodes"
    files = glob.glob(os.path.join(dataset_path, "**/*.json"), recursive=True)
    
    if len(files) == 0:
        print("No training data found! Check the file path.")
        return
        
    print(f"Found {len(files)} replays. Beginning chunked memory-safe training pipeline...")
    
    model = OrbitNet()
    optimizer = optim.Adam(model.parameters(), lr=0.001) # Lower LR for dataset stability
    
    # We don't use nn.MSELoss() here because we will implement a custom Weighted MSE below.
    
    chunk_size = 500
    epochs_per_chunk = 5 
    
    for i in range(0, len(files), chunk_size):
        chunk_files = files[i:i+chunk_size]
        print(f"\n=> Processing Replay Chunk[{i} to {i+len(chunk_files)}]")
        
        X_train, Y_train = extract_training_chunk(chunk_files)
        if len(X_train) == 0: continue
            
        print(f"   Extracted {len(X_train)} hindsight scenarios.")
        
        # Larger batch size to smooth out "lucky" dataset noise
        dataset = TensorDataset(X_train, Y_train)
        dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)
        
        for epoch in range(epochs_per_chunk): 
            total_loss = 0.0
            
            for batch_X, batch_Y in dataloader:
                optimizer.zero_grad()
                outputs = model(batch_X)
                
                # --- WEIGHTED LOSER POV MSE LOSS ---
                # Index 5 is 'Success Context' (1.0 if Winner, 0.0 if Loser). 
                # We double the MSE penalty if the model gets the loser's mistakes wrong.
                is_winner_flag = batch_X[:, 5:6] 
                weights = torch.where(is_winner_flag == 0.0, 2.0, 1.0)
                
                loss = (weights * (outputs - batch_Y) ** 2).mean()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                
            if epoch == epochs_per_chunk - 1:
                avg_loss = total_loss / len(dataloader)
                print(f"   Chunk Completed - Final Epoch Avg Weighted MSE: {avg_loss:.4f}")

    # --- Safe ONNX Export ---
    print("\nExporting OrbitNet V2 to ONNX...")
    model.eval() 
    dummy_input = torch.randn(1, 12) # Matches the 12 features
    export_path = "/kaggle/working/orbit_model.onnx"
    
    import onnx
    torch.onnx.export(
        model, dummy_input, export_path, export_params=True,
        opset_version=15, do_constant_folding=True,
        input_names=['input'], output_names=['output']
    )
    
    # Force IR Version 9 for Kaggle backend compatibility
    onnx_model = onnx.load(export_path)
    onnx_model.ir_version = 9
    onnx.save(onnx_model, export_path)
    
    print(f"Successfully saved Deep Hindsight IR-9 Model to: {export_path}")

if __name__ == "__main__":
    train_and_export()

Found 2631 replays. Beginning chunked memory-safe training pipeline...

=> Processing Replay Chunk[0 to 500]
   Extracted 55138 hindsight scenarios.
   Chunk Completed - Final Epoch Avg Weighted MSE: 0.0358

=> Processing Replay Chunk[500 to 1000]
   Extracted 59343 hindsight scenarios.
   Chunk Completed - Final Epoch Avg Weighted MSE: 0.0354

=> Processing Replay Chunk[1000 to 1500]
   Extracted 54157 hindsight scenarios.
   Chunk Completed - Final Epoch Avg Weighted MSE: 0.0343

=> Processing Replay Chunk[1500 to 2000]
   Extracted 58454 hindsight scenarios.
   Chunk Completed - Final Epoch Avg Weighted MSE: 0.0339

=> Processing Replay Chunk[2000 to 2500]
   Extracted 57045 hindsight scenarios.
   Chunk Completed - Final Epoch Avg Weighted MSE: 0.0340

=> Processing Replay Chunk[2500 to 2631]
   Extracted 14472 hindsight scenarios.
   Chunk Completed - Final Epoch Avg Weighted MSE: 0.0370

Exporting OrbitNet V2 to ONNX...


W0510 22:46:10.966000 22 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 15 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 15).
Failed to convert the model to the target version 15 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Successfully saved Deep Hindsight IR-9 Model to: /kaggle/working/orbit_model.onnx
